# Coffee17 Hong classification ablation
Validation-only factorial DSConv x SPPF-Attention pada EfficientNetV2-B0 + GAP. Test tetap terkunci.

In [ ]:
SEEDS = [123]
MODELS = ['HCD1', 'HCS1', 'HCDS1']
REPO_REF = 'agent/hong-classification-ablation'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
DRIVE_FOLDER = 'coffee17-hong-classification'
EVALUATION_SPLIT = 'val'

In [ ]:
# SETUP REPO, DRIVE, DATASET, DAN BASELINE
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
assert EVALUATION_SPLIT == 'val', 'Test dikunci pada fail-fast.'
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
dataset_base = Path('/content/coffee17-hong-data')
original, clean = dataset_base / 'original', dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
suffixes = {'.jpg', '.jpeg', '.png'}
def image_count(path):
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in suffixes) if path.is_dir() else 0
if image_count(original / 'source') != 979:
    if original.exists(): shutil.rmtree(original)
    if archive.exists() and not zipfile.is_zipfile(archive): archive.unlink()
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17', '--output', original, '--archive', archive, '--seed', '42'], cwd=repo)
audit_path = clean / 'audit.json'
if not (audit_path.is_file() and data_root.is_dir()):
    if clean.exists(): shutil.rmtree(clean)
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds', '--source-root', original / 'source', '--output-root', clean, '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1'], cwd=repo)
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))
token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan HF_TOKEN ke Colab Secrets.'
os.environ['HF_TOKEN'] = token
from huggingface_hub import snapshot_download
baseline_root = Path('/content/backbone-results')
patterns = []
for seed in SEEDS:
    patterns += [f'outputs/BE2G_seed{seed}/*', f'outputs/BE2H_seed{seed}/*']
snapshot_download(repo_id=HF_REPO, repo_type='model', token=token, local_dir=baseline_root, allow_patterns=patterns)
output_root = Path('/content/drive/MyDrive') / DRIVE_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('DATASET :', data_root)
print('OUTPUT  :', output_root)

In [ ]:
# PASTIKAN BASELINE GAP/HBP TERSEDIA DAN BUAT REPORT VALIDATION
baseline_command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_backbone_screening', '--data-root', str(data_root), '--output-root', str(baseline_root), '--seeds', *map(str, SEEDS), '--backbones', 'EV2', '--heads', 'gap', 'hbp', '--evaluation-split', 'val', '--hf-repo', HF_REPO, '--hf-sync-every', '2']
process = subprocess.Popen(baseline_command, cwd=repo)
while process.poll() is None:
    rows = []
    for code in ('BE2G', 'BE2H'):
        for seed in SEEDS:
            history = baseline_root / f'outputs/{code}_seed{seed}/history.json'
            if history.is_file():
                try: rows.append(f'{code}-s{seed}={len(json.loads(history.read_text()))}/50')
                except Exception: rows.append(f'{code}-s{seed}=saving')
    print('[BASELINE]', ', '.join(rows) if rows else 'restore/evaluasi', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Baseline gagal; lihat error di atas.'

In [ ]:
# TRAIN / RESUME HCD1, HCS1, HCDS1 DENGAN HEARTBEAT
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_hong_classification_screening', '--data-root', str(data_root), '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--seeds', *map(str, SEEDS), '--models', *MODELS, '--evaluation-split', EVALUATION_SPLIT]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
while process.poll() is None:
    rows = []
    for code in MODELS:
        for seed in SEEDS:
            history = output_root / f'outputs/{code}_seed{seed}/history.json'
            if history.is_file():
                try: rows.append(f'{code}-s{seed}={len(json.loads(history.read_text()))}/50')
                except Exception: rows.append(f'{code}-s{seed}=saving')
    print(f'[HEARTBEAT {(time.monotonic()-started)/60:.1f} menit]', ', '.join(rows) if rows else 'inisialisasi', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Runner gagal; lihat error di atas.'
print('SELESAI:', output_root)

In [ ]:
# TAMPILKAN PUTUSAN
decision_path = output_root / 'val_reports/hong_classification_decision.json'
assert decision_path.is_file(), f'Belum ada: {decision_path}'
report = json.loads(decision_path.read_text())
print('=== PUTUSAN HONG CLASSIFICATION ===')
for name, row in report['decisions'].items():
    print(f"{name:20s}: {row['decision']}")
print('Test dibuka:', report['test_opened'])
print('Klaim runtime DSConv:', report['dsconv_runtime_claim'])

## Setelah runtime reset
Jalankan ulang seluruh cell. Baseline dipulihkan dari Hugging Face dan seluruh checkpoint kandidat berada di `MyDrive/coffee17-hong-classification`. Jangan mengganti validation menjadi test.